# Document Summarization
Reads `iso27001.pdf` from this folder, chunks the text, and summarizes it using BART.

In [8]:
# ── Config ──────────────────────────────────────────────────────────────────
PDF_FILE     = "iso27001.pdf"   # must be in the same folder as this notebook
MODEL_NAME   = "facebook/bart-large-cnn"
CHUNK_TOKENS = 900    # BART's max input is 1024 — stay safely below it
MIN_TOKENS   = 50
MAX_TOKENS   = 150    # per-chunk summary length
FINAL_MAX    = 600    # final merged summary length

In [2]:
# ── Install (run once) ───────────────────────────────────────────────────────
# !uv pip install transformers torch pypdf

In [3]:
# ── Load model ───────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("Model ready.")

Loading facebook/bart-large-cnn ...


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Model ready.


In [4]:
# ── Extract text from PDF ────────────────────────────────────────────────────
import pypdf

reader = pypdf.PdfReader(PDF_FILE)
text   = "\n".join(page.extract_text() or "" for page in reader.pages)

print(f"Pages  : {len(reader.pages)}")
print(f"Words  : {len(text.split()):,}")
print(f"Preview: {text[:300]} ...")

Pages  : 26
Words  : 6,748
Preview: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001 ...


In [9]:
# ── Chunk text ───────────────────────────────────────────────────────────────
def chunk_text(text: str) -> list:
    words = text.split()
    chunks, current, length = [], [], 0
    for word in words:
        word_len = len(tokenizer.tokenize(word))
        if length + word_len > CHUNK_TOKENS:
            chunks.append(" ".join(current))
            current, length = [word], word_len
        else:
            current.append(word)
            length += word_len
    if current:
        chunks.append(" ".join(current))
    return chunks

chunks = chunk_text(text)
print(f"{len(chunks)} chunks created")

13 chunks created


In [10]:
# ── Summarize each chunk ─────────────────────────────────────────────────────
def summarize_chunk(text: str) -> str:
    inputs  = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(
        **inputs,
        min_length=MIN_TOKENS,
        max_length=MAX_TOKENS,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Summarizing {len(chunks)} chunks ...")
chunk_summaries = []
for i, chunk in enumerate(chunks):
    summary = summarize_chunk(chunk)
    chunk_summaries.append(summary)
    print(f"  [{i+1}/{len(chunks)}] done")

print("All chunks summarized.")

Summarizing 13 chunks ...
  [1/13] done
  [2/13] done
  [3/13] done
  [4/13] done
  [5/13] done
  [6/13] done
  [7/13] done
  [8/13] done
  [9/13] done
  [10/13] done
  [11/13] done
  [12/13] done
  [13/13] done
All chunks summarized.


In [11]:
# ── Merge into final summary ─────────────────────────────────────────────────
if len(chunk_summaries) == 1:
    final_summary = chunk_summaries[0]
else:
    merged  = " ".join(chunk_summaries)
    inputs  = tokenizer(merged, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(
        **inputs,
        min_length=MIN_TOKENS,
        max_length=FINAL_MAX,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True,
    )
    final_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(final_summary)

FINAL SUMMARY
 ISO/IEC 27000 describes the overview and the vocabulary of information security management systems. The adoption of an information security system is a strategic decision for an organization. The requirements set out in this document are generic and are intended to be applicable to all organizations, regardless of type, size or nature.
